In [1]:
# lectura de los datos

import pandas as pd
import read_data

# Suponiendo que el activo tiene ese nombre
# solo es remplazar por el nombre que se tenga
# solo se acepta archivos .parquet y .csv
nombre_activo: str = "EURUSD.parquet"
data: pd.DataFrame = read_data.read_asset(nombre_activo)

primer_lunes = data[data.index.dayofweek == 0].index[0]

data = data[primer_lunes:]

In [ ]:
import find_best
from read_data import ohlc_form
from use_tecnics import main, simple_methods
import keys

keys.methods = simple_methods
keys.candles = 20

comp_sem = pd.Timedelta(days=4)
next_lun = pd.Timedelta(days=3)
ventana_ent = pd.Timedelta(weeks=3) + comp_sem

inicio = data.index[0].date()
fin = data.index[-1].date()

while True:
    train_time_init = inicio
    train_time_fin = inicio + ventana_ent

    test_time_init = train_time_fin + next_lun
    test_time_fin = test_time_init + comp_sem

    inicio += comp_sem + next_lun
    if inicio >= fin:
        break

    print(f"Etapa de entrenamiento desde {train_time_init} hasta  {train_time_fin}")
    data_w = data.loc[train_time_init: train_time_fin]
    best_ma = find_best.opti_main(data_w)

    with open("Data/Moving_average.txt", "a") as archivo:
        archivo.write(f"{test_time_init} {test_time_fin}, {best_ma}\n")
    

Etapa de entrenamiento desde 2025-02-24 hasta  2025-03-21
Resultado obtenido entrenando desde 2025-02-24 hasta 2025-03-20
Método: MIDPOINT, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0.7580645161290323
risk reward: 0.5694704136459483
profit factor: 2.4331917673963246
trades: 62
Resultado de sobre ajuste 1.7107049375823418
Etapa de entrenamiento desde 2025-03-03 hasta  2025-03-28


In [3]:
import numpy as np
import pandas as pd

ma_week = []

with open("Data/Moving_average.txt", "r") as archivo:

    for line in archivo:
        start = pd.to_datetime(line[:10]).date()
        fin = pd.to_datetime(line[11:21]).date()

        lista = eval(line[22: -1], {"np": np})

        ma_week.append([start, fin, lista])

In [4]:
from use_tecnics import main

def get_signals_and_prices() -> pd.Series:
    signals_and_prices = None
    
    for week in ma_week:
        time_col = week[0] - pd.Timedelta(weeks=1)
        data_week = data.loc[time_col: week[1]]

        data_week = read_data.ohlc_form(data_week, f"{week[2][1]}min")["close"]
        data_week = main(week[2][0], data_week, week[2][2]).loc[week[0]: week[1]]
        if len(data_week) == 0:
            continue
            
        print(data_week)

        f_i = data_week.index[0]
        l_i = data_week.index[-1]

        if data_week["Signals"][f_i] == -1:
            f_i = data_week.index[1]

        if data_week["Signals"][l_i] == 1:
            f_i = data_week.index[-2]

        data_week = data_week.loc[f_i: l_i]

        if signals_and_prices is None:
            signals_and_prices = data_week
        else:
            signals_and_prices = pd.concat([signals_and_prices, data_week])

    return signals_and_prices

In [5]:
signals_and_prices = get_signals_and_prices()

                     Signals    Prices
time                                  
2025-03-24 01:36:00      1.0  1.081840
2025-03-24 02:08:00     -1.0  1.082350
2025-03-24 04:48:00      1.0  1.083050
2025-03-24 05:04:00     -1.0  1.082690
2025-03-24 05:52:00      1.0  1.082790
2025-03-24 06:08:00     -1.0  1.082860
2025-03-24 06:56:00      1.0  1.082155
2025-03-24 08:48:00     -1.0  1.083130
2025-03-24 12:16:00      1.0  1.083295
2025-03-24 12:48:00     -1.0  1.083940
2025-03-24 15:12:00      1.0  1.082650
2025-03-25 12:16:00     -1.0  1.079740
2025-03-25 18:56:00      1.0  1.080830
2025-03-25 19:12:00     -1.0  1.080920
2025-03-25 21:52:00      1.0  1.080150
2025-03-26 12:00:00     -1.0  1.079940
2025-03-26 13:04:00      1.0  1.080005
2025-03-26 13:20:00     -1.0  1.078850
2025-03-26 13:36:00      1.0  1.079190
2025-03-26 14:24:00     -1.0  1.079640
2025-03-26 15:12:00      1.0  1.079120
2025-03-26 17:20:00     -1.0  1.078990
2025-03-26 18:24:00      1.0  1.078580
2025-03-26 18:40:00     -

In [6]:
signals_and_prices

,Signals,Prices
time,,
2025-03-24 01:36:00,1.0,1.08184
2025-03-24 02:08:00,-1.0,1.08235
2025-03-24 04:48:00,1.0,1.08305
2025-03-24 05:04:00,-1.0,1.08269
2025-03-24 05:52:00,1.0,1.08279
...,...,...
2026-02-18 15:12:00,-1.0,1.18396
2026-02-18 15:28:00,1.0,1.18328
2026-02-19 08:32:00,-1.0,1.17979


In [8]:
from tester import backtest

backtest(signals_and_prices, False)

(np.float64(0.6461916461916462),
 np.float64(0.6838337674400783),
 np.float64(1.2576802855716125),
 407)

In [80]:
initial_mon = 1000
cantidad = 0
apalancamiento = 1

spread_pips = 0.0000

capital_maximo = initial_mon
drawdown_maximo = 0.0

for index in signals_and_prices.index:
    precio_actual = signals_and_prices["Prices"][index]
    current_signal = signals_and_prices["Signals"][index]

    if current_signal == 1 and cantidad == 0:
        poder_adquisitivo = initial_mon * apalancamiento
        
        precio_con_spread = precio_actual + spread_pips
        
        cantidad = int(poder_adquisitivo / precio_con_spread)

        if cantidad > 0:
            initial_mon -= (precio_con_spread * cantidad) 

    elif current_signal == -1 and cantidad > 0:
        initial_mon += (precio_actual * cantidad) 
        cantidad = 0

    equidad_actual = initial_mon + (precio_actual * cantidad) if cantidad > 0 else initial_mon
    
    if equidad_actual > capital_maximo:
        capital_maximo = equidad_actual
        
    drawdown_actual = ((capital_maximo - equidad_actual) / capital_maximo) * 100
    
    if drawdown_actual > drawdown_maximo:
        drawdown_maximo = drawdown_actual

print(f"Capital final CON SPREAD: ${initial_mon:.2f}")
print(f"Ganancia Neta: ${(initial_mon - 1000):.2f}")
print(f"Max Drawdown: {drawdown_maximo:.2f}%")

Capital final CON SPREAD: $1016.66
Ganancia Neta: $16.66
Max Drawdown: 4.98%
